# ECT Cosmology — Interactive

**Paper §12** | Hubble tension, JWST enhancement, inflation

$$G_{\rm eff}(z)=G_N(1+z)^{2\varepsilon},\quad\varepsilon\approx0.01\;\Rightarrow\;\Delta H_0\approx+3\,{\rm km/s/Mpc}$$

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.integrate import quad
from ipywidgets import interact, FloatSlider
%matplotlib inline

Om, OL = 0.315, 0.685
H0_Pl = 67.4; H0_loc = 73.0
km_Mpc = 3.0857e19; Gyr = 3.1557e16

def H_ect(z, H0, eps): return H0*np.sqrt(Om*(1+z)**3*(1+z)**(2*eps)+OL)
def age(H0, eps): return quad(lambda z: 1/((1+z)*H_ect(z,H0,eps)), 0, 30)[0]*km_Mpc/Gyr

def plot_cosmo(H0=67.4, eps=0.01):
    z = np.linspace(0,3,300)
    fig,axes = plt.subplots(1,3,figsize=(15,4))
    
    H_L = H0_Pl*np.sqrt(Om*(1+z)**3+OL)
    H_E = np.array([H_ect(zi,H0,eps) for zi in z])
    axes[0].plot(z,H_L,'b-',lw=2,label=f'ΛCDM H₀={H0_Pl}')
    axes[0].plot(z,H_E,'g--',lw=2,label=f'ECT H₀={H0:.1f}, ε={eps:.3f}')
    t_L=age(H0_Pl,0); t_E=age(H0,eps)
    axes[0].set_title(f'H(z) | Age: ΛCDM={t_L:.2f} Gyr, ECT={t_E:.2f} Gyr')
    axes[0].set_xlabel('z'); axes[0].set_ylabel('H [km/s/Mpc]')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
    
    z2 = np.linspace(0,15,300)
    axes[1].plot(z2,(1+z2)**(2*eps),'g-',lw=2)
    axes[1].axhline(1,'b',ls='--',lw=1.5,label='ΛCDM')
    axes[1].fill_between(z2,1,(1+z2)**(2*eps),alpha=0.2,color='g')
    axes[1].set_xlabel('z'); axes[1].set_ylabel(r'$G_{\rm eff}/G_N$')
    axes[1].set_title('G_eff(z) enhancement'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
    
    Z,Nu = np.meshgrid(np.linspace(4,16,50),np.linspace(2,7,50))
    F = (1+Z)**(2*eps); Enh = np.log10(np.clip(np.exp(Nu**2/2*(1-1/F)),1,1e6))
    im=axes[2].contourf(Z,Nu,Enh,levels=20,cmap='hot_r')
    plt.colorbar(im,ax=axes[2],label='log₁₀ enhancement')
    axes[2].contour(Z,Nu,Enh,levels=[0.5,1,2,3],colors='white',linewidths=0.8)
    axes[2].set_xlabel('z'); axes[2].set_ylabel('ν (peak height)')
    axes[2].set_title('JWST: Press-Schechter halo enhancement')
    plt.tight_layout(); plt.show()
    print(f'ΔH₀ = {H0-H0_Pl:+.1f} km/s/Mpc | Δage = {t_E-t_L:+.3f} Gyr | G_eff(z=10) = {(11)**(2*eps):.4f}×G_N')

interact(plot_cosmo,
  H0=FloatSlider(value=67.4,min=60,max=75,step=0.1,description='H₀:',continuous_update=False,
                 style={'description_width':'initial'}),
  eps=FloatSlider(value=0.01,min=0,max=0.05,step=0.001,description='ε:',continuous_update=False,
                  style={'description_width':'initial'}));

## Inflation spectral index

$$n_s = 1 - \frac{2}{N_e} = 0.967 \; (N_e=60) \quad\text{vs observed }0.965\pm0.004$$

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

def plot_inflation(Ne=60):
    ns = 1 - 2/Ne; r = 8/Ne
    Ne_range = np.arange(40, 80)
    ns_range = 1 - 2/Ne_range
    fig,ax = plt.subplots(figsize=(8,4))
    ax.plot(Ne_range, ns_range, 'g-', lw=2, label='ECT: $n_s=1-2/N_e$')
    ax.axhline(0.965, color='red', ls='--', lw=1.5, label='Planck 2018: 0.965±0.004')
    ax.fill_between(Ne_range, 0.961, 0.969, alpha=0.2, color='red')
    ax.axvline(Ne, color='green', ls=':', lw=2)
    ax.scatter([Ne],[ns],c='green',s=100,zorder=5)
    ax.set_xlabel('$N_e$ (e-folds)'); ax.set_ylabel('$n_s$')
    ax.set_title(f'N_e={Ne}: n_s={ns:.4f}, r={r:.4f}')
    ax.legend(fontsize=10); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    dev = abs(ns-0.965)/0.004
    print(f'n_s={ns:.4f} | r={r:.4f} | deviation from Planck: {dev:.1f}σ')

interact(plot_inflation, Ne=IntSlider(value=60,min=40,max=80,step=1,description='N_e:'));